# 02 · Marketing Performance Analysis: Lumora Home
Answers the business questions using the **cleaned** tables from notebook 01.
Reporting currency: **USD** (GBP rows converted at the monthly average rate in `fx_rates_monthly_clean.csv`; Feb-2026 rate is an estimate, flagged in cleaning).

**Two sources of truth are compared throughout**
- *Platform-reported*: `conversions` / `conversion_value` from the ad platforms (their own attribution).
- *Order-based*: real orders in `orders.csv`, credited to the **last-touch campaign** (and first-touch where stated). Revenue = `order_value - discount_amount` (before refunds, ex-shipping).

Outputs are written to `../outputs/` (analysis tables) and `../powerbi/` (star-schema files for the dashboard).

In [1]:
import pandas as pd, numpy as np, os
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
C = "../data_clean/"; OUT = "../outputs/"; PBI = "../powerbi/"
os.makedirs(OUT, exist_ok=True); os.makedirs(PBI, exist_ok=True)

camp = pd.read_csv(C+"campaigns_clean.csv", parse_dates=["start_date","end_date"])
ads  = pd.read_csv(C+"ad_performance_daily_clean.csv", parse_dates=["date"])
ords = pd.read_csv(C+"orders_clean.csv", parse_dates=["order_date"])
cust = pd.read_csv(C+"customers_clean.csv", parse_dates=["signup_date"])
fx   = pd.read_csv(C+"fx_rates_monthly_clean.csv", parse_dates=["month"])
print({n: d.shape for n, d in dict(campaigns=camp, ads=ads, orders=ords, customers=cust, fx=fx).items()})

{'campaigns': (111, 17), 'ads': (72935, 16), 'orders': (47385, 15), 'customers': (30142, 7), 'fx': (22, 3)}


## 1 · Build the analysis base: one currency, attributed orders

In [2]:
# --- 1a. Convert everything to USD using the month's average GBP->USD rate
rate = fx.set_index("month")["gbp_to_usd"]
def to_usd(df, date_col):
    # month start of each row's date, then look up the rate (USD rows use 1.0)
    df["month"] = df[date_col].dt.to_period("M").dt.to_timestamp()
    df["fx"] = np.where(df["currency"] == "GBP", df["month"].map(rate), 1.0)
    assert df["fx"].notna().all(), "a month has no FX rate"
    return df

ads  = to_usd(ads, "date")
ords = to_usd(ords, "order_date")
ads["spend_usd"] = ads["spend_clean"] * ads["fx"]          # spend_clean already fixes x100 errors
ads["rev_usd"]   = ads["conversion_value"] * ads["fx"]     # platform-reported revenue

ords["net_usd"] = (ords["order_value"] - ords["discount_amount"]) * ords["fx"]
ords["refunded"] = ords["is_refunded"].astype(str).str.lower().isin(["true","1","yes","y"])
ords["net_after_refund_usd"] = np.where(ords["refunded"], 0, ords["net_usd"])
ords["valid"] = ords["order_value_flag"].eq("ok")           # excludes zero/negative value orders

# --- 1b. Attach campaign attributes to ads and to each order's first / last touch
dim = camp.set_index("campaign_id")
ads = ads.merge(camp[["campaign_id","channel","platform","campaign_type","market","planned_budget"]],
                on="campaign_id", how="left")
for t in ("first","last"):
    key = ords[f"{t}_touch_campaign_id"]
    for a in ("channel","platform","campaign_type"):
        ords[f"{t}_{a}"] = key.map(dim[a]).fillna("Direct / none")

# --- 1c. Sanity checks (should all be quiet)
print("ads rows with no campaign match:", ads["channel"].isna().sum())
print("valid orders:", ords["valid"].sum(), "of", len(ords))
print("USD spend total: ", round(ads["spend_usd"].sum()))

ads rows with no campaign match: 0
valid orders: 47291 of 47385
USD spend total:  1753470


## Q1 · Is performance improving? Monthly trend (USD)

In [3]:
m = ads.groupby("month").agg(spend_usd=("spend_usd","sum"), clicks=("clicks","sum"),
                             platform_conversions=("conversions","sum"), platform_revenue_usd=("rev_usd","sum"))
o = ords[ords.valid].groupby("month").agg(orders=("order_id","count"), net_revenue_usd=("net_usd","sum"))
monthly = m.join(o)
monthly["platform_roas"]    = monthly.platform_revenue_usd / monthly.spend_usd
monthly["order_roas"]       = monthly.net_revenue_usd / monthly.spend_usd   # all orders incl. direct - blended
monthly["cost_per_order"]   = monthly.spend_usd / monthly.orders
monthly.round(2).to_csv(OUT+"q1_monthly_trend.csv")
monthly.round(2)

,spend_usd,clicks,platform_conversions,platform_revenue_usd,orders,net_revenue_usd,platform_roas,order_roas,cost_per_order
month,,,,,,,,,
2025-01-01,20955.61,40470,1166.87,95160.01,1149,93614.87,4.54,4.47,18.24
2025-02-01,32162.67,67893,1466.70,120299.63,1325,111313.71,3.74,3.46,24.27
2025-03-01,50448.99,88563,2196.62,181765.98,1843,156880.17,3.60,3.11,27.37
2025-04-01,53288.22,91442,2223.49,183631.63,1790,150316.86,3.45,2.82,29.77
2025-05-01,48358.29,91583,1998.88,163115.33,1702,143485.82,3.37,2.97,28.41
2025-06-01,50247.53,67321,1867.68,154114.19,1542,132338.25,3.07,2.63,32.59
2025-07-01,60767.55,78486,2484.25,200994.15,1949,165568.65,3.31,2.72,31.18
2025-08-01,65102.74,83692,2624.64,213844.14,2082,171066.40,3.28,2.63,31.27
2025-09-01,79569.26,99126,2432.26,197153.68,1990,169693.86,2.48,2.13,39.98


## Q2/Q3 · Channel and platform efficiency: platform-reported vs order-based

In [4]:
def efficiency(level_ads, level_ord):
    a = ads.groupby(level_ads).agg(spend_usd=("spend_usd","sum"), impressions=("impressions","sum"), clicks=("clicks","sum"),
                                   platform_conv=("conversions","sum"), platform_rev=("rev_usd","sum"))
    o = ords[ords.valid].groupby(level_ord).agg(orders=("order_id","count"), net_rev=("net_usd","sum"),
                                                net_rev_after_refund=("net_after_refund_usd","sum"))
    o.index.names = a.index.names
    t = a.join(o)
    t["ctr"] = t.clicks/t.impressions;       t["cpc"] = t.spend_usd/t.clicks
    t["platform_roas"] = t.platform_rev/t.spend_usd; t["order_roas"] = t.net_rev/t.spend_usd
    t["platform_cpa"]  = t.spend_usd/t.platform_conv; t["order_cpa"]  = t.spend_usd/t.orders
    t["platform_overcount_x"] = t.platform_conv/t.orders      # >1 => platform claims more purchases than real orders
    return t

by_channel  = efficiency("channel",  "last_channel")
by_platform = efficiency("platform", "last_platform")
by_type     = efficiency(["channel","campaign_type"], ["last_channel","last_campaign_type"])
for n, t in (("channel", by_channel), ("platform", by_platform), ("channel_campaign_type", by_type)):
    t.round(3).to_csv(OUT+f"q2_efficiency_by_{n}.csv")
print("Share of valid orders with NO attributed campaign (direct):",
      round((ords[ords.valid].last_channel=="Direct / none").mean(),3))
by_channel.round(2)

Share of valid orders with NO attributed campaign (direct): 0.167


,spend_usd,impressions,clicks,platform_conv,platform_rev,orders,net_rev,net_rev_after_refund,ctr,cpc,platform_roas,order_roas,platform_cpa,order_cpa,platform_overcount_x
channel,,,,,,,,,,,,,,,
Display,138705.70,90397880,677064,1967.72,155606.18,1299,106726.07,100916.29,0.01,0.20,1.12,0.77,70.49,106.78,1.51
Email,6723.91,4771294,157364,3025.00,235977.52,2212,179521.60,168471.81,0.03,0.04,35.10,26.70,2.22,3.04,1.37
Paid Search,634428.06,11895639,595756,25919.51,2133708.23,17424,1481429.77,1390187.39,0.05,1.06,3.36,2.34,24.48,36.41,1.49
Paid Social,474272.50,62747711,799772,10419.00,833223.17,6467,523318.91,489835.80,0.01,0.59,1.76,1.10,45.52,73.34,1.61
Shopping,456478.49,43617214,704563,17604.25,1414321.26,11785,973443.20,914374.44,0.02,0.65,3.10,2.13,25.93,38.73,1.49
Video,42861.81,31555004,160291,333.13,26554.17,219,18261.37,17449.99,0.01,0.27,0.62,0.43,128.66,195.72,1.52


In [5]:
by_platform.round(2)

,spend_usd,impressions,clicks,platform_conv,platform_rev,orders,net_rev,net_rev_after_refund,ctr,cpc,platform_roas,order_roas,platform_cpa,order_cpa,platform_overcount_x
platform,,,,,,,,,,,,,,,
Google Ads,1017392.90,53750192,1236770,42395.76,3457283.56,28389,2386738.40,2241073.16,0.02,0.82,3.40,2.35,24.00,35.84,1.49
Google Display Network,138705.70,90397880,677064,1967.72,155606.18,1299,106726.07,100916.29,0.01,0.20,1.12,0.77,70.49,106.78,1.51
Klaviyo,6723.91,4771294,157364,3025.00,235977.52,2212,179521.60,168471.81,0.03,0.04,35.10,26.70,2.22,3.04,1.37
Meta,430644.85,51340291,689699,9748.00,779481.93,5986,484002.60,452535.39,0.01,0.62,1.81,1.12,44.18,71.94,1.63
Microsoft Ads,73513.64,1762661,63549,1128.00,90745.93,820,68134.57,63488.67,0.04,1.16,1.23,0.93,65.17,89.65,1.38
Pinterest,24953.42,8641975,78038,572.00,45209.35,409,32675.40,30857.36,0.01,0.32,1.81,1.31,43.62,61.01,1.40
TikTok,18674.23,2765445,32035,99.00,8531.89,72,6640.91,6443.05,0.01,0.58,0.46,0.36,188.63,259.36,1.38
YouTube,42861.81,31555004,160291,333.13,26554.17,219,18261.37,17449.99,0.01,0.27,0.62,0.43,128.66,195.72,1.52


In [6]:
by_type.round(2)

spend_usd  impressions  clicks  platform_conv  platform_rev  orders    net_rev  net_rev_after_refund   ctr   cpc  platform_roas  order_roas  platform_cpa  order_cpa  platform_overcount_x
channel     campaign_type                                                                                                                                                                                            
Display     Prospecting    116576.53     80781797  606724        1066.36      82778.97     708   56786.93              53876.92  0.01  0.19           0.71        0.49        109.32     164.66                  1.51
            Retargeting     22129.16      9616083   70340         901.36      72827.21     591   49939.14              47039.37  0.01  0.31           3.29        2.26         24.55      37.44                  1.53
Email       CRM              6723.91      4771294  157364        3025.00     235977.52    2212  179521.60             168471.81  0.03  0.04          35.10       26.70          2.22       3.04                  1.37
Paid Search Brand           77022.50      1646794  186320       13993.33    1150136.03    9393  793033.84             745683.04  0.11  0.41          14.93       10.30          5.50       8.20                  1.49
            Non-Brand      557405.55     10248845  409436       11926.18     983572.20    8031  688395.93             644504.35  0.04  1.36           1.76        1.24         46.74      69.41                  1.49
Paid Social Prospecting    424926.26     58972344  717146        7741.00     624160.97    4788  392569.40             366379.28  0.01  0.59           1.47        0.92         54.89      88.75                  1.62
            Retargeting     49346.24      3775367   82626        2678.00     209062.20    1679  130749.51             123456.52  0.02  0.60           4.24        2.65         18.43      29.39                  1.59
Shopping    Shopping       456478.49     43617214  704563       17604.25    1414321.26   11785  973443.20             914374.44  0.02  0.65           3.10        2.13         25.93      38.73                  1.49
Video       Prospecting     42861.81     31555004  160291         333.13      26554.17     219   18261.37              17449.99  0.01  0.27           0.62        0.43        128.66     195.72                  1.52

## Q4 · Campaign level: who wastes money, who earns it

In [7]:
c = ads.groupby("campaign_id").agg(spend_usd=("spend_usd","sum"), impressions=("impressions","sum"), clicks=("clicks","sum"),
                                   conv=("conversions","sum"), rev=("rev_usd","sum"))
o = ords[ords.valid].groupby("last_touch_campaign_id").agg(orders=("order_id","count"), net_rev=("net_usd","sum"))
c = c.join(o).fillna({"orders":0,"net_rev":0}).join(dim[["campaign_name","channel","platform","campaign_type","market"]])
c["platform_roas"] = c.rev/c.spend_usd;  c["order_roas"] = c.net_rev/c.spend_usd
c["order_cpa"] = c.spend_usd/c.orders.replace(0, np.nan);  c["cvr"] = c.conv/c.clicks
c["spend_share"] = c.spend_usd/c.spend_usd.sum()
c.round(3).sort_values("spend_usd", ascending=False).to_csv(OUT+"q4_campaign_level.csv")

big = c[c.spend_usd >= 5000].sort_values("order_roas")
print("Worst 8 campaigns with >= $5k spend (order-based ROAS)"); display(big.head(8).round(2))
print("Best 5"); display(big.tail(5).round(2))
# Same-type comparison for the biggest losing line: Display Prospecting
dp = c[(c.channel=="Display")&(c.campaign_type=="Prospecting")].sort_values("spend_usd", ascending=False)
dp[["campaign_name","market","spend_usd","spend_share","clicks","cvr","order_roas","order_cpa"]].round(3)

Worst 8 campaigns with >= $5k spend (order-based ROAS)


,spend_usd,impressions,clicks,conv,rev,orders,net_rev,campaign_name,channel,platform,campaign_type,market,platform_roas,order_roas,order_cpa,cvr,spend_share
campaign_id,,,,,,,,,,,,,,,,,
CMP-0068,7421.71,2333235,23020,24.91,1948.94,18,1427.64,US_Video_Prospecting_Decor_202601,Video,YouTube,Prospecting,US,0.26,0.19,412.32,0.00,0.00
CMP-0066,7159.28,6633734,24831,32.51,2405.80,21,1722.22,UK_Video_Prospecting_Lighting_202601,Video,YouTube,Prospecting,UK,0.34,0.24,340.92,0.00,0.00
CMP-0042,18674.23,2765445,32035,99.00,8531.89,72,6640.91,us paidsocial prospecting spring 202509,Paid Social,TikTok,Prospecting,US,0.46,0.36,259.36,0.00,0.01
CMP-0092,72218.67,51896093,433285,624.40,46828.43,410,31525.88,UK_Display_Prospecting_Decor_202606,Display,Google Display Network,Prospecting,UK,0.65,0.44,176.14,0.00,0.04
CMP-0104,9644.56,6052103,39900,78.47,6651.91,53,4591.39,US_Video_Prospecting_Sale_202608,Video,YouTube,Prospecting,US,0.69,0.48,181.97,0.00,0.01
CMP-0050,14691.84,7306819,47807,133.91,11061.60,91,7473.89,US_Display_Prospecting_NewArrivals_202510,Display,Google Display Network,Prospecting,US,0.75,0.51,161.45,0.00,0.01
CMP-0096,7976.62,4644401,55750,72.03,5723.96,50,4517.32,UK_Display_Prospecting_Storage_202607,Display,Google Display Network,Prospecting,UK,0.72,0.57,159.53,0.00,0.00
CMP-0057,32273.73,3102806,47348,367.00,28113.99,233,18805.85,UK_PaidSocial_Prospecting_Lighting_202511,Paid Social,Meta,Prospecting,UK,0.87,0.58,138.51,0.01,0.02


Best 5


,spend_usd,impressions,clicks,conv,rev,orders,net_rev,campaign_name,channel,platform,campaign_type,market,platform_roas,order_roas,order_cpa,cvr,spend_share
campaign_id,,,,,,,,,,,,,,,,,
CMP-0077,12108.81,1671126,23952,951.20,80708.07,651,55936.61,US_Shopping_Shopping_Sofas_202603,Shopping,Google Ads,Shopping,US,6.67,4.62,18.60,0.04,0.01
CMP-0043,18720.18,284533,38498,1537.00,129922.83,1047,89271.31,US_PaidSearch_Brand_Brand_202509,Paid Search,Google Ads,Brand,US,6.94,4.77,17.88,0.04,0.01
CMP-0103,6549.05,92294,10816,733.71,61111.84,486,41294.92,US_PaidSearch_Brand_Brand_202608,Paid Search,Google Ads,Brand,US,9.33,6.31,13.48,0.07,0.00
CMP-0001,17043.45,527092,56818,3241.83,250969.25,2234,177586.80,UK_PaidSearch_Brand_Brand_202501,Paid Search,Google Ads,Brand,UK,14.73,10.42,7.63,0.06,0.01
CMP-0003,29305.41,619875,67509,7408.85,623499.58,4925,426505.52,US_PaidSearch_Brand_Brand_202501,Paid Search,Google Ads,Brand,US,21.28,14.55,5.95,0.11,0.02


,campaign_name,market,spend_usd,spend_share,clicks,cvr,order_roas,order_cpa
campaign_id,,,,,,,,
CMP-0092,UK_Display_Prospecting_Decor_202606,UK,72218.673,0.041,433285,0.001,0.437,176.143
CMP-0050,US_Display_Prospecting_NewArrivals_202510,US,14691.840,0.008,47807,0.003,0.509,161.449
CMP-0080,UK_Display_Prospecting_Rugs_202604,UK,11312.819,0.006,35581,0.003,0.624,137.961
CMP-0099,US_Display_Prospecting_Sale_202607,US,8384.280,0.005,20092,0.005,0.621,133.084
CMP-0096,UK_Display_Prospecting_Storage_202607,UK,7976.622,0.005,55750,0.001,0.566,159.532
CMP-0081,US_Display_Prospecting_NewArrivals_202604,US,1992.300,0.001,14209,0.001,0.501,166.025


## Q5 · Creative / audience fatigue: does CTR decay over a campaign's life?

In [8]:
rows = []
for cid, d in ads.groupby("campaign_id"):
    days = d.groupby("date")[["clicks","impressions"]].sum().sort_index()
    if len(days) < 60: continue                      # too short to judge
    k = len(days)//3                                 # compare first third vs last third of delivery days
    f, l = days.iloc[:k], days.iloc[-k:]
    rows.append((cid, len(days), f.clicks.sum()/f.impressions.sum(), l.clicks.sum()/l.impressions.sum()))
fat = pd.DataFrame(rows, columns=["campaign_id","delivery_days","ctr_first_third","ctr_last_third"]).set_index("campaign_id")
fat["ctr_change"] = fat.ctr_last_third/fat.ctr_first_third - 1
fat = fat.join(dim[["campaign_name","channel","platform","campaign_type"]])
fat.round(4).sort_values("ctr_change").to_csv(OUT+"q5_ctr_decay.csv")
print("Median change across campaigns:", round(fat.ctr_change.median(),3), "| campaigns:", len(fat))
fat.sort_values("ctr_change").head(10).round(3)

Median change across campaigns: 0.015 | campaigns: 65


,delivery_days,ctr_first_third,ctr_last_third,ctr_change,campaign_name,channel,platform,campaign_type
campaign_id,,,,,,,,
CMP-0050,115,0.012,0.004,-0.673,US_Display_Prospecting_NewArrivals_202510,Display,Google Display Network,Prospecting
CMP-0092,123,0.015,0.005,-0.658,UK_Display_Prospecting_Decor_202606,Display,Google Display Network,Prospecting
CMP-0075,108,0.016,0.007,-0.523,US_PaidSocial_Prospecting_Bedding_202603,Paid Social,Pinterest,Prospecting
CMP-0016,137,0.015,0.008,-0.462,US_PaidSocial_Prospecting_Bedding_202503,Paid Social,Meta,Prospecting
CMP-0021,110,0.012,0.007,-0.390,UK_PaidSocial_Prospecting_Spring_202505,Paid Social,Meta,Prospecting
CMP-0049,128,0.011,0.007,-0.359,US_PaidSocial_Prospecting_Sofas_202510,Paid Social,Meta,Prospecting
CMP-0037,76,0.043,0.032,-0.253,US_PaidSearch_NonBrand_Sofas_202508,Paid Search,Google Ads,Non-Brand
CMP-0053,77,0.017,0.012,-0.251,UK_Shopping_Shopping_Sofas_202510,Shopping,Google Ads,Shopping
CMP-0042,118,0.011,0.009,-0.231,us paidsocial prospecting spring 202509,Paid Social,TikTok,Prospecting


In [9]:
# Look at the decay shape of the worst campaigns: monthly CTR (does it fall steadily?)
worst = fat.sort_values("ctr_change").head(6).index
mc = (ads[ads.campaign_id.isin(worst)].groupby(["campaign_id", ads.date.dt.to_period("M")])
        [["clicks","impressions"]].sum().assign(ctr=lambda d: d.clicks/d.impressions)["ctr"].unstack(0))
mc.round(4)

campaign_id,CMP-0016,CMP-0021,CMP-0049,CMP-0050,CMP-0075,CMP-0092
date,,,,,,
2025-03,0.0121,NaN,NaN,NaN,NaN,NaN
2025-04,0.0168,NaN,NaN,NaN,NaN,NaN
2025-05,0.0100,0.0134,NaN,NaN,NaN,NaN
2025-06,0.0085,0.0066,NaN,NaN,NaN,NaN
2025-07,0.0088,0.0066,NaN,NaN,NaN,NaN
2025-08,0.0066,0.0074,NaN,NaN,NaN,NaN
2025-09,NaN,0.0073,NaN,NaN,NaN,NaN
2025-10,NaN,NaN,0.0112,0.0040,NaN,NaN
2025-11,NaN,NaN,0.0098,0.0126,NaN,NaN


## Q6 · Device performance

In [10]:
dev = ads[ads.device!="Unknown"].groupby(["channel","device"]).agg(spend_usd=("spend_usd","sum"), clicks=("clicks","sum"),
        conv=("conversions","sum"), rev=("rev_usd","sum"))
dev["cvr"]=dev.conv/dev.clicks; dev["cpa"]=dev.spend_usd/dev.conv; dev["roas"]=dev.rev/dev.spend_usd
dev["spend_share_in_channel"] = dev.spend_usd/dev.groupby("channel").spend_usd.transform("sum")
dev.round(3).to_csv(OUT+"q6_device.csv"); dev.round(3)

spend_usd  clicks      conv          rev    cvr      cpa    roas  spend_share_in_channel
channel     device                                                                                            
Display     Desktop   49051.648  130404    831.56    65908.197  0.006   58.988   1.344                   0.365
            Mobile    75798.989  464659    940.97    74611.269  0.002   80.554   0.984                   0.564
            Tablet     9588.792   60476    136.55    10518.940  0.002   70.222   1.097                   0.071
Email       Desktop    2381.487   45874   1292.00   100434.915  0.028    1.843  42.173                   0.362
            Mobile     3720.184   97190   1403.00   109407.473  0.014    2.652  29.409                   0.566
            Tablet      468.259   10410    241.00    18881.382  0.023    1.943  40.323                   0.071
Paid Search Desktop  225902.680  179536  10970.22   899518.526  0.061   20.592   3.982                   0.363
            Mobile   353305.283  362158  12688.57  1047283.138  0.035   27.844   2.964                   0.567
            Tablet    43561.826   42573   1783.98   147497.792  0.042   24.418   3.386                   0.070
Paid Social Desktop  167396.404  230871   4305.00   342079.071  0.019   38.884   2.044                   0.361
            Mobile   263481.972  482080   5192.00   417234.767  0.011   50.748   1.584                   0.568
            Tablet    32843.549   61467    663.00    52921.373  0.011   49.538   1.611                   0.071
Shopping    Desktop  160875.301  219784   7345.73   593014.626  0.033   21.901   3.686                   0.361
            Mobile   253269.214  418937   8616.43   688307.633  0.021   29.394   2.718                   0.568
            Tablet    31596.537   49913   1183.87    95611.801  0.024   26.689   3.026                   0.071
Video       Desktop   14439.167   48386    142.85    11390.531  0.003  101.079   0.789                   0.346
            Mobile    24488.440   71101    151.93    11913.175  0.002  161.182   0.486                   0.586
            Tablet     2864.002   24922     26.60     2367.730  0.001  107.669   0.827                   0.069

## Q7 · First-touch vs last-touch: who starts journeys, who closes them?

In [11]:
v = ords[ords.valid]
ft = v.groupby("first_channel").agg(first_touch_orders=("order_id","count"), first_touch_rev=("net_usd","sum"))
lt = v.groupby("last_channel").agg(last_touch_orders=("order_id","count"), last_touch_rev=("net_usd","sum"))
att = ft.join(lt, how="outer").fillna(0)
att["last_minus_first_orders"] = att.last_touch_orders - att.first_touch_orders
att["last_to_first_ratio"] = att.last_touch_orders/att.first_touch_orders
att.round(2).to_csv(OUT+"q7_first_vs_last_touch.csv"); att.round(2)

,first_touch_orders,first_touch_rev,last_touch_orders,last_touch_rev,last_minus_first_orders,last_to_first_ratio
first_channel,,,,,,
Direct / none,3704,305416.26,7885,654457.14,4181,2.13
Display,1385,111927.85,1299,106726.07,-86,0.94
Email,153,12912.41,2212,179521.60,2059,14.46
Paid Search,17385,1471484.55,17424,1481429.77,39,1.00
Paid Social,8467,699205.28,6467,523318.91,-2000,0.76
Shopping,14994,1240331.66,11785,973443.20,-3209,0.79
Video,1203,95880.06,219,18261.37,-984,0.18


## Q8 · CAC and customer quality (new customers only)

In [12]:
# A customer's FIRST valid order = their acquisition order. Orders without customer_id are excluded (can't be tracked).
x = v.dropna(subset=["customer_id"]).sort_values(["order_date","order_id"]).copy()
x["order_n"] = x.groupby("customer_id").cumcount()+1
first = x[x.order_n==1]
spend_ch = ads.groupby("channel").spend_usd.sum()

cac = first.groupby("first_channel").size().rename("new_customers_first_touch").to_frame()
cac["spend_usd"] = spend_ch.reindex(cac.index)
cac["cac_first_touch"] = cac.spend_usd/cac.new_customers_first_touch
cac["new_customers_last_touch"] = first.groupby("last_channel").size()
cac["cac_last_touch"] = cac.spend_usd/cac.new_customers_last_touch

cl = x.groupby("customer_id").agg(orders=("order_id","count"), net_rev=("net_usd","sum"),
                                  acq_channel=("first_channel","first"))
q = cl.groupby("acq_channel").agg(customers=("orders","count"), avg_orders=("orders","mean"),
                                  avg_net_rev_usd=("net_rev","mean"), repeat_rate=("orders", lambda s:(s>1).mean()))
cacq = cac.join(q)
cacq["rev_per_customer_to_cac"] = cacq.avg_net_rev_usd/cacq.cac_first_touch
cacq.round(3).to_csv(OUT+"q8_cac_and_customer_quality.csv")
print("Share of valid orders that are a customer's first:", round((x.order_n==1).mean(),3))
print("Data window is 22 months, so repeat metrics are right-censored for recent customers.")
cacq.round(2)

Share of valid orders that are a customer's first: 0.638
Data window is 22 months, so repeat metrics are right-censored for recent customers.


,new_customers_first_touch,spend_usd,cac_first_touch,new_customers_last_touch,cac_last_touch,customers,avg_orders,avg_net_rev_usd,repeat_rate,rev_per_customer_to_cac
first_channel,,,,,,,,,,
Direct / none,2142,NaN,NaN,3578,NaN,2142,1.72,141.67,0.40,NaN
Display,1163,138705.70,119.27,707,196.19,1163,1.19,95.82,0.16,0.80
Email,122,6723.91,55.11,225,29.88,122,1.25,105.46,0.18,1.91
Paid Search,11042,634428.06,57.46,12046,52.67,11042,1.57,132.81,0.37,2.31
Paid Social,5694,474272.50,83.29,4496,105.49,5694,1.48,122.33,0.33,1.47
Shopping,9068,456478.49,50.34,8783,51.97,9068,1.65,136.17,0.39,2.71
Video,797,42861.81,53.78,193,222.08,797,1.51,120.08,0.28,2.23


## Q9 · Does the CRM acquisition channel agree with order attribution?

In [13]:
cmap = first[["customer_id","first_channel"]].merge(cust[["customer_id","crm_acquisition_channel"]], on="customer_id", how="left")
agree = pd.crosstab(cmap.crm_acquisition_channel.fillna("Missing"), cmap.first_channel, normalize="index").round(2)
agree.to_csv(OUT+"q9_crm_vs_first_touch.csv"); agree

first_channel,Direct / none,Display,Email,Paid Search,Paid Social,Shopping,Video
crm_acquisition_channel,,,,,,,
Direct / Organic,0.60,0.02,0.00,0.12,0.10,0.14,0.02
Display,0.00,0.67,0.00,0.07,0.12,0.12,0.02
Email,0.00,0.07,0.57,0.07,0.14,0.14,0.02
Paid Search,0.00,0.02,0.00,0.74,0.10,0.12,0.02
Paid Social,0.00,0.03,0.00,0.13,0.68,0.14,0.03
Shopping,0.00,0.02,0.00,0.12,0.10,0.73,0.03
Unknown,0.07,0.05,0.00,0.39,0.18,0.29,0.02
Video,0.00,0.02,0.00,0.12,0.09,0.11,0.67


## Q10 · Budget pacing (planned vs actual, campaign currency)

In [14]:
b = ads.groupby("campaign_id").spend_clean.sum().rename("actual_spend").to_frame().join(
        camp.set_index("campaign_id")[["campaign_name","channel","status","planned_budget","budget_missing","currency"]])
b = b[~b.budget_missing].copy()
b["pacing"] = b.actual_spend/b.planned_budget
b.round(3).sort_values("pacing", ascending=False).to_csv(OUT+"q10_budget_pacing.csv")
print(b.pacing.describe().round(2))
print("Over plan:", (b.pacing>1).sum(), "| over 125%:", (b.pacing>1.25).sum(), "| under 85%:", (b.pacing<0.85).sum(), "| total:", len(b))
print("Ended/Paused campaigns under 85% are unspent money; Active ones are still running, so judge pacing by status.")
b.groupby("status").pacing.describe().round(2)

count    105.00
mean       0.97
std        0.18
min        0.70
25%        0.79
50%        0.94
75%        1.12
max        1.59
Name: pacing, dtype: float64
Over plan: 47 | over 125%: 3 | under 85%: 41 | total: 105
Ended/Paused campaigns under 85% are unspent money; Active ones are still running, so judge pacing by status.


,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Active,15.0,0.97,0.20,0.70,0.78,1.02,1.15,1.22
Ended,88.0,0.97,0.18,0.73,0.80,0.93,1.12,1.59
Paused,2.0,0.91,0.07,0.86,0.89,0.91,0.93,0.96


## Q11 · Seasonality: Black Friday window (15 Nov – 1 Dec 2025)

In [15]:
a25 = ads[ads.date.dt.year==2025].copy()
a25["period"] = np.where(((a25.date.dt.month==11)&(a25.date.dt.day>=15))|((a25.date.dt.month==12)&(a25.date.dt.day==1)), "Black Friday window", "Other days")
s = a25.groupby("period").agg(days=("date","nunique"), spend_usd=("spend_usd","sum"), clicks=("clicks","sum"), conv=("conversions","sum"), rev=("rev_usd","sum"))
s["spend_per_day"]=s.spend_usd/s.days; s["cvr"]=s.conv/s.clicks; s["platform_roas"]=s.rev/s.spend_usd
bf_o = ords[ords.valid & (ords.order_date.dt.year==2025)].copy()
bf_o["period"]=np.where(((bf_o.order_date.dt.month==11)&(bf_o.order_date.dt.day>=15))|((bf_o.order_date.dt.month==12)&(bf_o.order_date.dt.day==1)),"Black Friday window","Other days")
s["orders_per_day"] = bf_o.groupby("period").size()/s.days
s["avg_discount_pct"] = bf_o.groupby("period").apply(lambda d: d.discount_amount.sum()/d.order_value.sum())
s["order_roas"] = bf_o.groupby("period").net_usd.sum()/s.spend_usd
s.round(3).to_csv(OUT+"q11_black_friday.csv"); s.round(3)

,days,spend_usd,clicks,conv,rev,spend_per_day,cvr,platform_roas,orders_per_day,avg_discount_pct,order_roas
period,,,,,,,,,,,
Black Friday window,17,128551.398,162613,4217.84,344657.415,7561.847,0.026,2.681,177.059,0.111,1.826
Other days,348,778525.948,1086034,27010.51,2203607.193,2237.144,0.025,2.830,63.158,0.022,2.382


## Q12 · Tracking reconciliation: platform conversions vs real orders, and the Meta outage

In [16]:
pm = ads.groupby(["month","platform"]).agg(platform_conv=("conversions","sum"), spend_usd=("spend_usd","sum")).reset_index()
om = (ords[ords.valid].groupby(["month","last_platform"]).size().rename("orders").reset_index()
        .rename(columns={"last_platform":"platform"}))
rec = pm.merge(om, on=["month","platform"], how="left").fillna({"orders":0})
rec["conv_per_order"] = rec.platform_conv/rec.orders.replace(0, np.nan)
rec.round(3).to_csv(OUT+"q12_monthly_reconciliation.csv", index=False)
print("Meta conversions-per-order by month (steady ~1.6 except one month):")
rec[rec.platform=="Meta"][["month","platform_conv","orders","conv_per_order"]].round(2)

Meta conversions-per-order by month (steady ~1.6 except one month):


/tmp/ipykernel_169/2182927670.py:6: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  rec.round(3).to_csv(OUT+"q12_monthly_reconciliation.csv", index=False)
/tmp/ipykernel_169/2182927670.py:8: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  rec[rec.platform=="Meta"][["month","platform_conv","orders","conv_per_order"]].round(2)


,month,platform_conv,orders,conv_per_order
1,2025-01-01,13.0,7.0,1.86
4,2025-02-01,70.0,50.0,1.40
9,2025-03-01,149.0,85.0,1.75
14,2025-04-01,479.0,297.0,1.61
17,2025-05-01,363.0,215.0,1.69
21,2025-06-01,258.0,155.0,1.66
25,2025-07-01,621.0,362.0,1.72
29,2025-08-01,426.0,273.0,1.56
34,2025-09-01,410.0,249.0,1.65
40,2025-10-01,365.0,228.0,1.60


In [17]:
# Find days where a platform spent money but reported ZERO conversions - a tracking-outage signature.
day = ads.groupby(["date","platform"]).agg(conv=("conversions","sum"), spend_usd=("spend_usd","sum")).reset_index()
z = day[(day.conv==0)&(day.spend_usd>100)].sort_values(["platform","date"])
# group consecutive days into runs
z["run"] = (z.groupby("platform").date.diff().dt.days.fillna(99) != 1).cumsum()
runs = z.groupby(["platform","run"]).agg(start=("date","min"), end=("date","max"), days=("date","count"), spend_usd=("spend_usd","sum"))
runs = runs[runs.days>=5]
display(runs.round(0))
# Compare with real orders credited to Meta in the same window
w0, w1 = runs.loc["Meta","start"].iloc[0], runs.loc["Meta","end"].iloc[0]
oi = ords[ords.valid & (ords.last_platform=="Meta") & ords.order_date.between(w0,w1)]
adw = ads[(ads.platform=="Meta") & ads.date.between(w0,w1)]
outage = pd.DataFrame({"window":[f"{w0.date()} to {w1.date()}"], "meta_spend_usd":[adw.spend_usd.sum()],
                       "meta_reported_conversions":[adw.conversions.sum()], "meta_attributed_orders_in_orders_table":[len(oi)],
                       "net_revenue_usd_of_those_orders":[oi.net_usd.sum()]}).round(0)
outage.to_csv(OUT+"q12_meta_outage.csv", index=False); outage

/tmp/ipykernel_169/805441073.py:8: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(runs.round(0))


,,start,end,days,spend_usd
platform,run,,,,
Meta,22,2026-03-10,2026-03-24,15,5047.0
TikTok,60,2025-12-15,2025-12-19,5,926.0
YouTube,77,2026-02-05,2026-02-09,5,653.0


,window,meta_spend_usd,meta_reported_conversions,meta_attributed_orders_in_orders_table,net_revenue_usd_of_those_orders
0,2026-03-10 to 2026-03-24,5047.0,0.0,118,8989.0


## Power BI data model (star schema)
Simple, clean files: one fact table per grain plus shared dimensions. All money fields are USD; original currency kept for reference.

In [18]:
# --- dim_date
d = pd.DataFrame({"date": pd.date_range("2025-01-01","2026-10-31")})
d["year"]=d.date.dt.year; d["month_start"]=d.date.dt.to_period("M").dt.to_timestamp(); d["month_name"]=d.date.dt.strftime("%b %Y")
d["quarter"]="Q"+d.date.dt.quarter.astype(str)+" "+d.date.dt.year.astype(str); d["weekday"]=d.date.dt.day_name()
d["is_black_friday_window"]=(((d.date.dt.year==2025)&(d.date.dt.month==11)&(d.date.dt.day>=15))|((d.date.dt.year==2025)&(d.date.dt.month==12)&(d.date.dt.day==1)))
d["is_meta_tracking_outage"]=d.date.between("2026-03-10","2026-03-24")
d.to_csv(PBI+"dim_date.csv", index=False)

# --- dim_campaign
dc = camp[["campaign_id","campaign_name","channel","platform","campaign_type","objective","market","start_date","end_date","planned_budget","currency","status","owner_team","budget_missing"]]
dc.to_csv(PBI+"dim_campaign.csv", index=False)

# --- fact_ad_performance (daily grain, USD)
fa = ads[ads.channel.notna()][["date","campaign_id","ad_group_id","device","impressions","clicks","conversions","currency","fx","spend_clean","spend_usd","conversion_value","rev_usd",
        "spend_flag","clicks_exceed_impressions"]].rename(columns={"spend_clean":"spend_local","conversion_value":"platform_revenue_local","rev_usd":"platform_revenue_usd"})
fa.round({"spend_usd":4,"platform_revenue_usd":4}).to_csv(PBI+"fact_ad_performance.csv", index=False)

# --- dim_customer with acquisition (first valid order) channel
ac = first[["customer_id","order_date","first_channel","last_channel"]].rename(columns={"order_date":"first_order_date","first_channel":"acquisition_channel_first_touch","last_channel":"acquisition_channel_last_touch"})
dcu = cust.merge(ac, on="customer_id", how="left")
dcu.drop(columns=["email_hash"]).to_csv(PBI+"dim_customer.csv", index=False)

# --- fact_orders with attribution and new-customer flag
fo = ords.merge(first[["order_id"]].assign(is_first_order=True), on="order_id", how="left")
fo["is_first_order"]=fo.is_first_order.fillna(False)
fo = fo[["order_id","customer_id","order_date","country","currency","fx","product_category","order_value","discount_amount","shipping_cost","refunded","net_usd","net_after_refund_usd",
         "first_touch_campaign_id","last_touch_campaign_id","first_channel","last_channel","last_platform","last_campaign_type","valid","is_first_order"]]
fo.round({"net_usd":4,"net_after_refund_usd":4}).to_csv(PBI+"fact_orders.csv", index=False)
fx.to_csv(PBI+"dim_fx_rates.csv", index=False)
for f in sorted(os.listdir(PBI)): print(f, os.path.getsize(PBI+f)//1024, "KB")

dim_campaign.csv 16 KB
dim_customer.csv 2257 KB
dim_date.csv 41 KB
dim_fx_rates.csv 0 KB
fact_ad_performance.csv 6354 KB
fact_orders.csv 7228 KB


In [19]:
# --- Reconciliation checks: the Power BI facts must tie back to the analysis
assert abs(pd.read_csv(PBI+"fact_ad_performance.csv").spend_usd.sum() - ads.spend_usd.sum()) < 1
assert pd.read_csv(PBI+"fact_orders.csv").valid.sum() == ords.valid.sum()
assert abs(by_channel.spend_usd.sum() - ads.spend_usd.sum()) < 1
print("Checks passed. Total USD spend:", round(ads.spend_usd.sum()), "| valid orders:", int(ords.valid.sum()))

Checks passed. Total USD spend: 1753470 | valid orders: 47291
